# 04 – Water Body / Flood Segmentation (Computer Vision)
**AquaML Portfolio Project**

Semantic segmentation of water from imagery using a lightweight U-Net style model.

Because large aerial datasets (FloodNet, etc.) are heavy, this notebook:
1. Generates realistic synthetic RGB patches with water masks (so it runs everywhere)
2. Trains a small U-Net
3. Shows the full training + evaluation loop you would use on real satellite/aerial data

**For a real portfolio**: swap the synthetic generator with a few hundred FloodNet or Sentinel-2 patches.


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from src.evaluation import dice_coefficient, iou_score
from src.visualization import plot_segmentation_overlay

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


## Synthetic water-body dataset generator

In [ ]:
def generate_water_patch(size=128, seed=None):
    """Create a simple RGB image + binary water mask."""
    rng = np.random.default_rng(seed)
    # Background (land / vegetation)
    img = rng.normal(80, 25, (size, size, 3)).clip(0, 255).astype(np.uint8)
    # Add some texture
    img = (img * 0.7 + rng.integers(0, 40, (size, size, 3))).clip(0, 255).astype(np.uint8)

    mask = np.zeros((size, size), dtype=np.uint8)
    # Random water regions (ellipses / blobs)
    n_blobs = rng.integers(1, 4)
    for _ in range(n_blobs):
        cy, cx = rng.integers(20, size-20, 2)
        ry, rx = rng.integers(15, 45, 2)
        y, x = np.ogrid[:size, :size]
        blob = ((y - cy)/ry)**2 + ((x - cx)/rx)**2 <= 1
        mask[blob] = 1
        # Water color (blue-ish)
        img[blob] = [rng.integers(20, 60), rng.integers(60, 120), rng.integers(140, 220)]

    return img, mask

# Generate a small dataset
N = 400
images, masks = [], []
for i in range(N):
    im, m = generate_water_patch(seed=i)
    images.append(im)
    masks.append(m)
images = np.stack(images)
masks = np.stack(masks)
print(images.shape, masks.shape)


In [ ]:
# Visualize a few
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i in range(4):
    axes[0, i].imshow(images[i])
    axes[0, i].axis("off")
    axes[1, i].imshow(masks[i], cmap="Blues")
    axes[1, i].axis("off")
axes[0, 0].set_title("RGB patches")
axes[1, 0].set_title("Water masks")
plt.tight_layout()
plt.show()


## Lightweight U-Net

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)

class UNetTiny(nn.Module):
    def __init__(self, in_ch=3, out_ch=1):
        super().__init__()
        self.down1 = DoubleConv(in_ch, 32)
        self.pool1 = nn.MaxPool2d(2)
        self.down2 = DoubleConv(32, 64)
        self.pool2 = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(64, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv2 = DoubleConv(128, 64)
        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.conv1 = DoubleConv(64, 32)
        self.out = nn.Conv2d(32, out_ch, 1)
    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(self.pool1(d1))
        bn = self.bottleneck(self.pool2(d2))
        u2 = self.up2(bn)
        u2 = self.conv2(torch.cat([u2, d2], dim=1))
        u1 = self.up1(u2)
        u1 = self.conv1(torch.cat([u1, d1], dim=1))
        return torch.sigmoid(self.out(u1))

model = UNetTiny().to(device)
print(sum(p.numel() for p in model.parameters()) / 1e6, "M parameters")


In [ ]:
class WaterDataset(Dataset):
    def __init__(self, images, masks):
        self.images = images.astype(np.float32) / 255.0
        self.masks = masks.astype(np.float32)
    def __len__(self):
        return len(self.images)
    def __getitem__(self, idx):
        x = torch.from_numpy(self.images[idx]).permute(2, 0, 1)
        y = torch.from_numpy(self.masks[idx]).unsqueeze(0)
        return x, y

split = int(0.8 * N)
train_ds = WaterDataset(images[:split], masks[:split])
test_ds  = WaterDataset(images[split:], masks[split:])
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=16)

opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.BCELoss()

model.train()
for epoch in range(15):
    total = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        pred = model(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        opt.step()
        total += loss.item() * len(xb)
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:2d} | loss {total/len(train_ds):.4f}")


In [ ]:
# Evaluation
model.eval()
dice_scores, iou_scores = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        pred = (model(xb) > 0.5).cpu().numpy()
        true = yb.numpy()
        for p, t in zip(pred, true):
            dice_scores.append(dice_coefficient(t, p))
            iou_scores.append(iou_score(t, p))
print(f"Mean Dice: {np.mean(dice_scores):.3f}")
print(f"Mean IoU : {np.mean(iou_scores):.3f}")


In [ ]:
# Visual examples
model.eval()
xb, yb = next(iter(test_loader))
with torch.no_grad():
    pred = (model(xb.to(device)) > 0.5).cpu().numpy()
for i in range(3):
    img = (xb[i].permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    plot_segmentation_overlay(img, yb[i, 0].numpy(), pred[i, 0])
    plt.show()


## Production note
On real data (FloodNet, Sentinel-2, PlanetScope):
- Use stronger backbones (ResNet / EfficientNet encoder via `segmentation_models_pytorch`)
- Heavy augmentation (albumentations)
- Class imbalance handling for rare flood classes
- Multi-temporal / multi-spectral inputs
